# 02 · Preprocesamiento ECoG, operación por operación

Entrada: datos crudos de desarrollo y YAML. **El EDA es opcional**: no se leen sus carpetas ni sus figuras.
Este notebook prepara los tres sujetos con la misma cadena, por bloques independientes.
La purga pendiente se usa solo como candidato de verificación y nunca se aprueba automáticamente.

Al finalizar se publica `data/processed/<pipeline_id>/`. Copia ese identificador a
`paths.processed_pipeline_id` en el YAML para que el 03 verifique exactamente estos datos.
Cada ejecución tiene su propio `run_id`; el pipeline se identifica por su contrato y contenido numérico.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

## Entorno
Se instalan solo las dependencias que falten (`requirements.txt`: incluye MNE y scikit-learn). No se instala PyTorch ni CUDA. Si un módulo cambia después de importarlo, reinicia el runtime.

In [ ]:
env_check = subprocess.run([sys.executable, str(PROJECT / 'scripts/check_environment.py'), '--install-missing'],
                           capture_output=True, text=True)
print(env_check.stdout + env_check.stderr)  # Colab no muestra la salida de subprocesos sin captura.
env_check.check_returncode()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '->', tt_ecog.__file__)

## Drive
Montar Drive en Colab. Las rutas experimentales y todos los parámetros se leen del YAML.

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
CONFIG = PROJECT / 'configs/experiment_config.yaml'

## 1. Configuración y lectura de desarrollo

In [ ]:
import numpy as np
import pandas as pd
from tt_ecog import preprocessing as pp
from tt_ecog.prep import start_run, load_subjects, publish_pipeline, finish_run
from tt_ecog.prep.blocks import save_block, write_processing_metadata
from tt_ecog.prep.folds import subject_blocks, save_fold
from tt_ecog.splits import chronological_blocks, make_folds, train_eligibility
from tt_ecog.datasets import fit_fold_scalers, train_window_starts, inference_window_starts

ctx = start_run(CONFIG, DRIVE_ROOT)
config = ctx.config
records = load_subjects(ctx)
print('Corrida:', ctx.run_id, '| destino provisional:', ctx.staging)
display(pd.DataFrame(ctx.subject_status).T)

## 2. Particiones y segmentos independientes
Se fijan B1–B5 antes de transformar. Cada entrada conserva sus límites crudos y sus coordenadas a 100 Hz.
Los bloques no se concatenan para filtrar, interpolar ni construir ventanas.

In [ ]:
names = config.data['validation']['block_names']
specs = {s: pp.PreprocessingSpec.from_config(config, s) for s in records}
blocks = {s: chronological_blocks(ecog.values.shape[1], names, specs[s].factor)
          for s, (ecog, glove) in records.items()}
segments = {}
for subject, subject_blocks_list in blocks.items():
    for block in subject_blocks_list:
        segments[subject, block.name] = {
            'block': block,
            'signal': records[subject][0].values[:, block.start_source:block.stop_source]
        }
display(pd.DataFrame([{'subject': s, 'block': name, 'start_s': v['block'].start_source / specs[s].sfreq,
                       'stop_s': v['block'].stop_source / specs[s].sfreq, 'shape': v['signal'].shape}
                      for (s, name), v in segments.items()]))

## 3. Exclusión fija de canales
La lista sale del YAML. Se aplica antes de la referencia y de los filtros.

In [ ]:
for (subject, name), segment in segments.items():
    segment['signal'] = pp.exclude_channels(segment['signal'], specs[subject].channels)
display(pd.DataFrame([{'subject': s, 'excluded': spec.channels.excluded,
                       'kept_original': spec.channels.kept, 'effective_channels': spec.channels.n_kept}
                      for s, spec in specs.items()]))

## 4. Referencia de mediana común
Mediana espacial por instante, sin estadísticas temporales aprendidas.

In [ ]:
reference_rows = []
for (subject, name), segment in segments.items():
    segment['signal'] = pp.common_median_reference(segment['signal'])
    reference_rows.append({'subject': subject, 'block': name,
                           'max_abs_spatial_median': float(np.abs(np.median(segment['signal'], axis=0)).max())})
display(pd.DataFrame(reference_rows))

## 5. FIR pasabanda y notch
Se muestran los diseños efectivos. Se aplica el kernel combinado centrado usado por la implementación
verificada; aplicar dos filtros separados con dos paddings podría cambiar los bordes.

In [ ]:
display(pd.DataFrame([{'subject': s, 'filter': design.name, 'taps': len(design.taps),
                       'half_support': design.half_support}
                      for s, spec in specs.items() for design in (spec.bandpass, spec.notch, spec.fir)]))
for (subject, name), segment in segments.items():
    segment['signal'] = pp.apply_fir(segment['signal'], specs[subject].fir)
print('Filtrado independiente terminado:', len(segments), 'bloques')

## 6. Morlet → potencia → antialias → decimación
Se ejecutan explícitamente Morlet y decimación por canal para controlar memoria. Nunca se usa el
`decim` de MNE. No se recortan los negativos pequeños producidos por el rizado FIR.

In [ ]:
for (subject, name), segment in segments.items():
    spec = specs[subject]
    signal = segment.pop('signal')
    n_out = (signal.shape[1] + spec.factor - 1) // spec.factor
    features = np.empty((signal.shape[0], len(spec.freqs), n_out), dtype=np.float32)
    negative = 0
    for channel in range(signal.shape[0]):
        power = pp.morlet_power(signal[channel:channel + 1], spec.sfreq, spec.freqs,
                                spec.n_cycles, spec.zero_mean)[0]
        downsampled = pp.decimate(power, spec.antialias, spec.factor)
        negative += int(np.count_nonzero(downsampled < 0))
        features[channel] = downsampled
    if not np.isfinite(features).all():
        raise ValueError(f'{subject} {name}: potencia no finita')
    segment['features'] = features
    segment['xvalid'] = pp.feature_valid_mask(signal.shape[1], spec)
    segment['report'] = {'negative_values': negative, 'total_values': int(features.size)}
    print(subject, name, features.shape, '| columnas con contexto:', int(segment['xvalid'].sum()))
    del signal, power, downsampled, features

## 7. Guante: recuperar 25 Hz e interpolar dentro de cada bloque

In [ ]:
gloves = {}
results = {}
for subject, (ecog, glove) in records.items():
    spec = specs[subject]
    inspection = pp.glove_storage_inspection(glove.values, glove.metadata['sampling_hz_glove_storage'], spec)
    y25, knots = pp.effective_glove(glove.values, inspection, spec)
    gloves[subject] = (y25, knots)
    results[subject] = {'spec': spec, 'blocks': blocks[subject], 'inspection': inspection,
                        'n_effective_glove': int(knots.size)}
    print(subject, inspection['method'], '| fase:', inspection['phase'], '| muestras:', knots.size)
for (subject, name), segment in segments.items():
    y25, knots = gloves[subject]
    block = segment['block']
    segment['targets'], segment['yvalid'] = pp.segment_glove(
        y25, knots, block.start_source, block.stop_source, specs[subject])

## 8. Retardo y máscara de parejas válidas
ECoG(t) → flexión(t + 200 ms). Se conservan índices globales de ambos tiempos.

In [ ]:
for (subject, name), segment in segments.items():
    spec = specs[subject]
    xa, ya, xi, yi = pp.align_delay(segment['features'], segment['targets'], spec.delay_samples)
    valid = segment['xvalid'][xi] & segment['yvalid'][yi]
    if not valid.any() or not np.isfinite(ya[:, valid]).all():
        raise ValueError(f'{subject} {name}: parejas inválidas')
    segment['aligned'] = (xa, ya, xi, yi)
    print(subject, name, '| retardo:', np.unique(yi - xi), '| parejas válidas:', int(valid.sum()))

## 9. Guardado provisional de bloques
Todavía no se publica un pipeline: faltan folds, escaladores y ventanas.

In [ ]:
block_rows = []
for (subject, name), segment in segments.items():
    xa, ya, xi, yi = segment['aligned']
    manifest = save_block(specs[subject], segment['block'], gloves[subject][1],
                          segment['features'], segment['xvalid'], segment['yvalid'],
                          xa, ya, xi, yi, segment['report'], ctx.staging / subject / name)
    block_rows.append({'subject': subject, 'block': name, 'shape': manifest['x']['shape'],
                        'valid_columns': manifest['columns']['valid']})
write_processing_metadata(config, records, results, ctx.staging)
ctx.save_table(pd.DataFrame(block_rows), 'blocks', 'Dimensiones y parejas válidas por bloque')
display(pd.DataFrame(block_rows))
del segments, segment, xa, ya

## 10. Folds y purga
La purga efectiva sigue pendiente cuando el YAML contiene `null`; aquí se etiqueta el candidato.

In [ ]:
folds = {}
for subject in records:
    subject_block_list, arrays = subject_blocks(ctx.staging, subject, names)
    for fold in make_folds(subject_block_list, config.data['validation']['stage2_validation_blocks']):
        eligible = {b.name: train_eligibility(arrays[b.name]['x_index'], arrays[b.name]['y_index'],
                      arrays[b.name]['valid'], b, fold.val, ctx.purge) for b in fold.train}
        folds[subject, fold.name] = {'fold': fold, 'arrays': arrays, 'eligible': eligible}
print('Purga:', ctx.purge, '|', ctx.purge_status, '| folds:', len(folds))

## 11. Escaladores ajustados exclusivamente con entrenamiento

In [ ]:
for (subject, name), item in folds.items():
    feature_scaler, target_scaler, stats = fit_fold_scalers(
        config, item['arrays'], item['eligible'], f'{subject} {name}')
    item.update(feature_scaler=feature_scaler, target_scaler=target_scaler, stats=stats)
display(pd.DataFrame([{'subject': s, 'fold': name, **item['stats']} for (s, name), item in folds.items()]))

## 12. Ventanas y transformación
Los índices se calculan dentro de tramos continuos. Se comprueba una ventana de validación con los
escaladores ya ajustados. Al cargar un fold se transforma cada bloque con esos mismos coeficientes;
no se duplican en disco las características escaladas de los cinco folds.

In [ ]:
v = config.data['validation']
for (subject, name), item in folds.items():
    fold, arrays, eligible = item['fold'], item['arrays'], item['eligible']
    train_windows = []
    for block in fold.train:
        starts = train_window_starts(eligible[block.name], v['train_window_samples'], v['train_stride_samples'])
        train_windows.append(np.column_stack((np.full(starts.size, names.index(block.name)), starts)))
    item['train_windows'] = np.concatenate(train_windows).astype(np.int64)
    item['val_windows'] = inference_window_starts(arrays[fold.val.name]['valid'],
                          v['inference_window_samples'], v['inference_stride_samples'])
    start = item['val_windows'][0]
    sample = arrays[fold.val.name]['x'][:, :, start:start + v['inference_window_samples']]
    scaled = item['feature_scaler'].transform(pp.features_to_matrix(sample))
    target = item['target_scaler'].transform(arrays[fold.val.name]['y'][:, start:start + v['inference_window_samples']].T)
    if not np.isfinite(scaled).all() or not np.isfinite(target).all():
        raise ValueError(f'{subject} {name}: ventana escalada no finita')
    print(subject, name, '| train:', len(item['train_windows']), '| validación:', len(item['val_windows']))

## 13. Persistir folds y publicar una identidad compartida

In [ ]:
fold_rows = []
for (subject, name), item in folds.items():
    fold_rows.append(save_fold(config, subject, item['fold'], item['arrays'], names,
                    list(specs[subject].channels.kept), ctx.purge, item['eligible'],
                    item['feature_scaler'], item['target_scaler'], item['stats'],
                    item['train_windows'], item['val_windows'], ctx.staging / 'folds' / subject / name))
ctx.save_table(pd.DataFrame(fold_rows), 'folds', 'Cobertura, ventanas y separación por fold')
display(pd.DataFrame(fold_rows))
# Liberar mapas de memoria antes de renombrar, también en Windows.
del folds, item, arrays, sample, scaled, target
pipeline_id = publish_pipeline(ctx)
summary = finish_run(ctx)
print('Estado:', summary['status'])
print('pipeline_id:', pipeline_id)
print('Copiar este valor a paths.processed_pipeline_id en el YAML y hacer commit/push antes de ejecutar 03.')
print('Publicar estos datos no equivale a aprobar V01–V14.')